# MedNoise: MRI denoising with DnCNN

This notebook trains a residual DnCNN model to remove synthetic Rician noise from grayscale MRI images. The model is evaluated with PSNR and SSIM and compared with Gaussian and median filtering.

The project was originally developed for a university course in 2025 and was later reorganized to make the experiment reproducible.


## 1. Environment setup

Google Colab with a GPU runtime is recommended. The dataset is downloaded automatically from Kaggle unless `MEDNOISE_DATA_DIR` points to an existing local copy.


In [ ]:
%pip install -q kagglehub opencv-python-headless scikit-image


In [ ]:
import hashlib
import os
import random
import time
from collections import defaultdict
from pathlib import Path

import cv2
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image
from IPython.display import display
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from torch.utils.data import DataLoader, Dataset, Subset


## 2. Configuration and reproducibility

All experiment settings are kept in one place. A fixed seed makes the dataset split and evaluation noise reproducible.


In [ ]:
SEED = 42

KAGGLE_DATASET_ID = "masoudnickparvar/brain-tumor-mri-dataset/versions/2"
DATA_DIR_OVERRIDE = os.getenv("MEDNOISE_DATA_DIR")

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 5e-4
MIN_LEARNING_RATE = 1e-5
EARLY_STOPPING_PATIENCE = 4
NOISE_TYPE = "rician"  # "rician" or "gaussian"
NOISE_SIGMA = 0.10
VALIDATION_RATIO = 0.15
NUM_EXAMPLES = 4

IN_COLAB = Path("/content").exists()
DEFAULT_OUTPUT_DIR = "/content/MedNoise" if IN_COLAB else "outputs"
OUTPUT_DIR = Path(os.getenv("MEDNOISE_OUTPUT_DIR", DEFAULT_OUTPUT_DIR)).resolve()
CHECKPOINT_PATH = OUTPUT_DIR / "dncnn_best.pth"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = 2 if IN_COLAB else 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Outputs: {OUTPUT_DIR}")


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


seed_everything(SEED)
dataloader_generator = torch.Generator().manual_seed(SEED)


## 3. Dataset preparation

By default, the notebook downloads version 2 of the [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) by Masoud Nickparvar, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

The dataset provides separate `Training` and `Testing` directories. The official test set remains separate, while a deterministic validation subset is taken only from the training directory. If those directories are unavailable, the notebook falls back to a deterministic 70/15/15 split.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


def list_images(root: Path) -> list[Path]:
    return sorted(
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def resolve_dataset_root() -> Path:
    if DATA_DIR_OVERRIDE:
        root = Path(DATA_DIR_OVERRIDE).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"MEDNOISE_DATA_DIR does not exist: {root}")
        return root

    print(f"Downloading Kaggle dataset: {KAGGLE_DATASET_ID}")
    return Path(kagglehub.dataset_download(KAGGLE_DATASET_ID)).resolve()


def split_dataset(root: Path) -> tuple[list[Path], list[Path], list[Path]]:
    training_dir = root / "Training"
    testing_dir = root / "Testing"
    rng = random.Random(SEED)

    if training_dir.is_dir() and testing_dir.is_dir():
        training_pool = list_images(training_dir)
        test_files = list_images(testing_dir)
        rng.shuffle(training_pool)

        validation_count = max(1, round(len(training_pool) * VALIDATION_RATIO))
        val_files = training_pool[:validation_count]
        train_files = training_pool[validation_count:]
    else:
        all_files = list_images(root)
        rng.shuffle(all_files)
        train_end = int(0.70 * len(all_files))
        val_end = train_end + int(0.15 * len(all_files))
        train_files = all_files[:train_end]
        val_files = all_files[train_end:val_end]
        test_files = all_files[val_end:]

    if not train_files or not val_files or not test_files:
        raise RuntimeError("The dataset split is empty. Check the dataset directory structure.")

    return train_files, val_files, test_files


DATASET_ROOT = resolve_dataset_root()
train_files, val_files, test_files = split_dataset(DATASET_ROOT)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Train: {len(train_files)} | Validation: {len(val_files)} | Test: {len(test_files)}")


## 4. Synthetic-noise dataset

Training samples receive fresh random noise, horizontal flips, and small rotations. Validation and test samples use path-derived seeds, so the same image receives the same synthetic noise across runs.


In [ ]:
class MRIDenoisingDataset(Dataset):
    def __init__(
        self,
        files: list[Path],
        img_size: int,
        noise_type: str,
        sigma: float,
        augment: bool,
        fixed_noise: bool,
    ) -> None:
        if noise_type not in {"rician", "gaussian"}:
            raise ValueError("noise_type must be 'rician' or 'gaussian'")

        self.files = files
        self.noise_type = noise_type
        self.sigma = sigma
        self.augment = augment
        self.fixed_noise = fixed_noise

        self.preprocess = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.Grayscale(num_output_channels=1),
            transforms.ToTensor(),
        ])
        self.augmentation = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
        ])

    @staticmethod
    def _seed_from_path(path: Path) -> int:
        stable_parts = path.parts[-3:] if len(path.parts) >= 3 else path.parts
        stable_key = "/".join(stable_parts)
        digest = hashlib.sha256(stable_key.encode("utf-8")).hexdigest()
        return int(digest[:8], 16)

    def _generator(self, path: Path, offset: int = 0) -> torch.Generator | None:
        if not self.fixed_noise:
            return None
        generator = torch.Generator()
        generator.manual_seed(self._seed_from_path(path) ^ offset)
        return generator

    def _random_noise(self, clean: torch.Tensor, path: Path, offset: int = 0) -> torch.Tensor:
        return torch.randn(
            clean.shape,
            dtype=clean.dtype,
            generator=self._generator(path, offset),
        )

    def _add_noise(self, clean: torch.Tensor, path: Path) -> torch.Tensor:
        first_noise = self._random_noise(clean, path)

        if self.noise_type == "gaussian":
            noisy = clean + first_noise * self.sigma
        else:
            second_noise = self._random_noise(clean, path, offset=0xABCDEF)
            noisy = torch.sqrt(
                (clean + first_noise * self.sigma) ** 2
                + (second_noise * self.sigma) ** 2
            )

        return noisy.clamp(0.0, 1.0)

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor, str]:
        path = self.files[index]
        with Image.open(path) as image:
            clean = self.preprocess(image.convert("L"))

        if self.augment:
            clean = self.augmentation(clean)

        noisy = self._add_noise(clean, path)
        return noisy, clean, str(path)


In [ ]:
train_dataset = MRIDenoisingDataset(
    train_files, IMG_SIZE, NOISE_TYPE, NOISE_SIGMA, augment=True, fixed_noise=False
)
val_dataset = MRIDenoisingDataset(
    val_files, IMG_SIZE, NOISE_TYPE, NOISE_SIGMA, augment=False, fixed_noise=True
)
test_dataset = MRIDenoisingDataset(
    test_files, IMG_SIZE, NOISE_TYPE, NOISE_SIGMA, augment=False, fixed_noise=True
)


def make_loader(dataset: Dataset, shuffle: bool) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=NUM_WORKERS > 0,
        worker_init_fn=seed_worker,
        generator=dataloader_generator,
    )


train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)


## 5. DnCNN model

DnCNN learns a residual noise map rather than the clean image directly. The denoised output is obtained by subtracting the predicted noise from the noisy input.


In [ ]:
class DnCNN(nn.Module):
    def __init__(self, depth: int = 17, features: int = 32, image_channels: int = 1) -> None:
        super().__init__()

        layers: list[nn.Module] = [
            nn.Conv2d(image_channels, features, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),
        ]

        for _ in range(depth - 2):
            layers.extend([
                nn.Conv2d(features, features, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(features),
                nn.ReLU(inplace=True),
            ])

        layers.append(
            nn.Conv2d(features, image_channels, kernel_size=3, padding=1, bias=True)
        )
        self.network = nn.Sequential(*layers)

    def forward(self, noisy: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        predicted_noise = self.network(noisy)
        denoised = noisy - predicted_noise
        return denoised, predicted_noise


model = DnCNN().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=MIN_LEARNING_RATE
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 6. Training and validation

The learning rate follows a cosine schedule, and training stops early if validation PSNR does not improve for four consecutive epochs. The best checkpoint is selected by validation PSNR. Mixed precision is enabled only when CUDA is available. Metrics are aggregated per image, avoiding bias from a smaller final batch.


In [ ]:
def image_metrics(
    denoised: torch.Tensor,
    clean: torch.Tensor,
) -> tuple[list[float], list[float]]:
    denoised_np = denoised.detach().cpu().numpy()
    clean_np = clean.detach().cpu().numpy()
    psnr_values, ssim_values = [], []

    for index in range(denoised_np.shape[0]):
        prediction = np.clip(denoised_np[index, 0], 0.0, 1.0)
        target = np.clip(clean_np[index, 0], 0.0, 1.0)
        psnr_values.append(peak_signal_noise_ratio(target, prediction, data_range=1.0))
        ssim_values.append(structural_similarity(target, prediction, data_range=1.0))

    return psnr_values, ssim_values


def train_one_epoch(model: nn.Module, loader: DataLoader) -> float:
    model.train()
    total_loss = 0.0

    for noisy, clean, _ in loader:
        noisy = noisy.to(DEVICE, non_blocking=True)
        clean = clean.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            denoised, _ = model(noisy)
            loss = criterion(denoised, clean)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * noisy.size(0)

    return total_loss / len(loader.dataset)


@torch.no_grad()
def validate(model: nn.Module, loader: DataLoader) -> tuple[float, float, float]:
    model.eval()
    total_loss = 0.0
    psnr_values, ssim_values = [], []

    for noisy, clean, _ in loader:
        noisy = noisy.to(DEVICE, non_blocking=True)
        clean = clean.to(DEVICE, non_blocking=True)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            denoised, _ = model(noisy)
            loss = criterion(denoised, clean)

        total_loss += loss.item() * noisy.size(0)
        batch_psnr, batch_ssim = image_metrics(denoised, clean)
        psnr_values.extend(batch_psnr)
        ssim_values.extend(batch_ssim)

    return (
        total_loss / len(loader.dataset),
        float(np.mean(psnr_values)),
        float(np.mean(ssim_values)),
    )


In [ ]:
history = {
    "epoch": [],
    "learning_rate": [],
    "train_loss": [],
    "val_loss": [],
    "val_psnr": [],
    "val_ssim": [],
    "epoch_sec": [],
}
best_psnr = float("-inf")
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    started_at = time.perf_counter()
    learning_rate = optimizer.param_groups[0]["lr"]
    train_loss = train_one_epoch(model, train_loader)
    val_loss, val_psnr, val_ssim = validate(model, val_loader)
    elapsed = time.perf_counter() - started_at

    history["epoch"].append(epoch)
    history["learning_rate"].append(learning_rate)
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_psnr"].append(val_psnr)
    history["val_ssim"].append(val_ssim)
    history["epoch_sec"].append(elapsed)

    print(
        f"[{epoch:02d}/{EPOCHS}] lr={learning_rate:.2e} | "
        f"train_loss={train_loss:.6f} | val_loss={val_loss:.6f} | "
        f"val_PSNR={val_psnr:.2f} dB | val_SSIM={val_ssim:.4f} | "
        f"time={elapsed:.1f}s"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        epochs_without_improvement = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  Saved new best checkpoint: {CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1

    scheduler.step()

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(
            f"Early stopping after {EARLY_STOPPING_PATIENCE} epochs "
            "without validation PSNR improvement."
        )
        break


history_frame = pd.DataFrame(history)
history_path = OUTPUT_DIR / "training_log.csv"
history_frame.to_csv(history_path, index=False)
print(f"Training log: {history_path}")


## 7. Training curves

The raw per-epoch values are plotted directly because smoothing can hide real behavior and create misleading edge effects in a short training run.


In [ ]:
CURVES_DIR = OUTPUT_DIR / "curves"
CURVES_DIR.mkdir(parents=True, exist_ok=True)
epochs = history_frame["epoch"]


def save_curve(columns: list[str], ylabel: str, title: str, filename: str) -> None:
    plt.figure(figsize=(7, 4.5))
    for column in columns:
        plt.plot(epochs, history_frame[column], marker="o", label=column)
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(epochs)
    plt.grid(True, linestyle="--", alpha=0.3)
    if len(columns) > 1:
        plt.legend()
    plt.tight_layout()
    plt.savefig(CURVES_DIR / filename, dpi=150, bbox_inches="tight")
    plt.show()


save_curve(["train_loss", "val_loss"], "MSE loss", "Training and validation loss", "loss_curves.png")
save_curve(["val_psnr"], "PSNR (dB)", "Validation PSNR", "val_psnr_curve.png")
save_curve(["val_ssim"], "SSIM", "Validation SSIM", "val_ssim_curve.png")


## 8. Test-set evaluation

The best DnCNN checkpoint is compared with Gaussian and median filtering on the official test set.


In [ ]:
@torch.no_grad()
def evaluate_dncnn(model: nn.Module, loader: DataLoader) -> tuple[float, float]:
    model.eval()
    psnr_values, ssim_values = [], []

    for noisy, clean, _ in loader:
        noisy = noisy.to(DEVICE, non_blocking=True)
        clean = clean.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            denoised, _ = model(noisy)
        batch_psnr, batch_ssim = image_metrics(denoised, clean)
        psnr_values.extend(batch_psnr)
        ssim_values.extend(batch_ssim)

    return float(np.mean(psnr_values)), float(np.mean(ssim_values))


@torch.no_grad()
def evaluate_classical_filters(
    loader: DataLoader,
    gaussian_kernel: int = 5,
    gaussian_sigma: float = 1.0,
    median_kernel: int = 5,
) -> tuple[tuple[float, float], tuple[float, float]]:
    gaussian_psnr, gaussian_ssim = [], []
    median_psnr, median_ssim = [], []

    for noisy, clean, _ in loader:
        noisy_np = noisy.numpy()
        clean_np = clean.numpy()

        for index in range(noisy_np.shape[0]):
            noisy_uint8 = (noisy_np[index, 0] * 255).astype(np.uint8)
            target = clean_np[index, 0]
            gaussian = cv2.GaussianBlur(
                noisy_uint8, (gaussian_kernel, gaussian_kernel), gaussian_sigma
            ).astype(np.float32) / 255.0
            median = cv2.medianBlur(noisy_uint8, median_kernel).astype(np.float32) / 255.0

            gaussian_psnr.append(peak_signal_noise_ratio(target, gaussian, data_range=1.0))
            gaussian_ssim.append(structural_similarity(target, gaussian, data_range=1.0))
            median_psnr.append(peak_signal_noise_ratio(target, median, data_range=1.0))
            median_ssim.append(structural_similarity(target, median, data_range=1.0))

    return (
        (float(np.mean(gaussian_psnr)), float(np.mean(gaussian_ssim))),
        (float(np.mean(median_psnr)), float(np.mean(median_ssim))),
    )


best_model = DnCNN().to(DEVICE)
best_model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))

dncnn_psnr, dncnn_ssim = evaluate_dncnn(best_model, test_loader)
(gaussian_psnr, gaussian_ssim), (median_psnr, median_ssim) = evaluate_classical_filters(test_loader)

test_metrics = pd.DataFrame([
    {"method": "Gaussian", "psnr": gaussian_psnr, "ssim": gaussian_ssim},
    {"method": "Median", "psnr": median_psnr, "ssim": median_ssim},
    {"method": "DnCNN", "psnr": dncnn_psnr, "ssim": dncnn_ssim},
])
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
display(test_metrics.round(4))


## 9. Qualitative comparison

One deterministic example is selected from each available diagnostic category, up to `NUM_EXAMPLES`. The metrics CSV stores paths relative to the dataset root instead of machine-specific cache paths.


In [ ]:
def select_diverse_indices(files: list[Path], count: int) -> list[int]:
    groups: dict[str, list[int]] = defaultdict(list)
    for index, path in enumerate(files):
        groups[path.parent.name].append(index)

    rng = random.Random(SEED)
    selected = [rng.choice(groups[label]) for label in sorted(groups)]

    if len(selected) < count:
        remaining = [index for index in range(len(files)) if index not in selected]
        rng.shuffle(remaining)
        selected.extend(remaining[: count - len(selected)])

    return selected[:count]


example_indices = select_diverse_indices(test_files, NUM_EXAMPLES)
example_loader = DataLoader(
    Subset(test_dataset, example_indices),
    batch_size=1,
    shuffle=False,
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)


In [ ]:
@torch.no_grad()
def save_qualitative_examples(
    model: nn.Module,
    loader: DataLoader,
    gaussian_kernel: int = 5,
    gaussian_sigma: float = 1.0,
    median_kernel: int = 5,
) -> pd.DataFrame:
    model.eval()
    samples_dir = OUTPUT_DIR / "samples"
    samples_dir.mkdir(parents=True, exist_ok=True)
    rows = []

    for sample_number, (noisy, clean, paths) in enumerate(loader, start=1):
        noisy_device = noisy.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            denoised, _ = model(noisy_device)

        prediction = denoised[0, 0].detach().clamp(0, 1).cpu().numpy()
        noisy_image = noisy[0, 0].numpy()
        target = clean[0, 0].numpy()
        noisy_uint8 = (noisy_image * 255).astype(np.uint8)

        gaussian = cv2.GaussianBlur(
            noisy_uint8, (gaussian_kernel, gaussian_kernel), gaussian_sigma
        ).astype(np.float32) / 255.0
        median = cv2.medianBlur(noisy_uint8, median_kernel).astype(np.float32) / 255.0

        metrics = {
            "gaussian_psnr": peak_signal_noise_ratio(target, gaussian, data_range=1.0),
            "gaussian_ssim": structural_similarity(target, gaussian, data_range=1.0),
            "median_psnr": peak_signal_noise_ratio(target, median, data_range=1.0),
            "median_ssim": structural_similarity(target, median, data_range=1.0),
            "dncnn_psnr": peak_signal_noise_ratio(target, prediction, data_range=1.0),
            "dncnn_ssim": structural_similarity(target, prediction, data_range=1.0),
        }

        figure, axes = plt.subplots(1, 5, figsize=(16, 3.5))
        panels = [
            (target, "Clean"),
            (noisy_image, "Noisy"),
            (gaussian, f"Gaussian\nPSNR={metrics['gaussian_psnr']:.2f} | SSIM={metrics['gaussian_ssim']:.4f}"),
            (median, f"Median\nPSNR={metrics['median_psnr']:.2f} | SSIM={metrics['median_ssim']:.4f}"),
            (prediction, f"DnCNN\nPSNR={metrics['dncnn_psnr']:.2f} | SSIM={metrics['dncnn_ssim']:.4f}"),
        ]

        for axis, (image, title) in zip(axes, panels):
            axis.imshow(image, cmap="gray", vmin=0, vmax=1)
            axis.set_title(title, fontsize=9)
            axis.axis("off")

        figure.tight_layout()
        figure_name = f"example_{sample_number:02d}.png"
        figure.savefig(samples_dir / figure_name, dpi=150, bbox_inches="tight")
        plt.show()
        plt.close(figure)

        source_path = Path(paths[0])
        try:
            source_file = source_path.relative_to(DATASET_ROOT).as_posix()
        except ValueError:
            source_file = source_path.name

        rows.append({
            "source_file": source_file,
            "category": source_path.parent.name,
            "figure_file": figure_name,
            **metrics,
        })

    metrics_frame = pd.DataFrame(rows)
    metrics_frame.to_csv(OUTPUT_DIR / "sample_metrics.csv", index=False)
    return metrics_frame


sample_metrics = save_qualitative_examples(best_model, example_loader)
display(sample_metrics.round(4))
print(f"All generated artifacts are available in: {OUTPUT_DIR}")


## 10. Limitations and next steps

- Noise is added synthetically; performance on real acquisition noise may differ.
- The model operates on independent 2D slices and does not use 3D anatomical context.
- PSNR and SSIM do not replace clinical evaluation.
- Future experiments could compare multiple noise levels and report variation across several random seeds.
